# ARC-AGI-3 — Chronos v19: black-box FORGE agent (competition-correct)

**What this ships.** v19 is the *black-box* agent built for the real scored surface.
The Kaggle private set runs in **competition mode: API-only, the game source is
unreachable, RHAE scoring** (`level_score = (human_actions / ai_actions)²` — every
action you spend costs you). That kills any white-box search (BFS / transfer all
need the game source). So v19 ships the **competition-legal** path that the
preview *winners* actually used:

- **`forge_agent.py`** — `ForgeAgent`: a CNN (`ChangeNet`) that predicts which
  action changes the frame (StochasticGoose, 1st) + a frame-hash transition graph
  with frontier exploration (Blind Squirrel, 2nd / graph-exploration, 3rd).
- **`pretrained_weights.pt`** — the offline-trained ChangeNet prior. This is the
  *only* thing learned offline, and it transfers as **knowledge in the weights**,
  never as stored answers. A warm change/novelty prior = far fewer wasted
  exploration actions = higher RHAE.
- **`combined_agent.py`** — entry point (`class MyAgent`). It first tries a
  white-box BFS; on the hidden eval no source is reachable, so it routes the whole
  game through the pretrained `ForgeAgent`.

---

## Setup (one-time)

Upload a **private Kaggle dataset** (e.g. named `v19-forge`) containing EXACTLY
these three files at the top level:

- `combined_agent.py`  — the entry point (`class MyAgent`)
- `forge_agent.py`     — the black-box `ForgeAgent` (CNN + graph)
- `pretrained_weights.pt` — the offline ChangeNet prior (**required** for real performance)

Then in this notebook: **Add Input** → attach that dataset **and** the competition
data. **Accelerator: GPU (T4)**. **Internet: OFF**.

### INTEGRITY CHECKLIST — the dataset must NOT contain
- `solutions/` (the BFS answer-book) or any `*_bfs_cache_*.json`
- engine sources / `v13`/`v15` scratch packages

The run is forced **honest** with `V19_STORE_SOLUTIONS=0`: the agent solves every
level live and never looks up a pre-computed answer. Cell 2 hard-asserts the
answer-book is absent before you can submit.

---

## Runtime behavior on the hidden eval (per game, blind)
1. **BFS probe fails instantly** — no game source on the API-only gateway (expected!).
2. The game is routed to the pretrained **`ForgeAgent`** (`prior=loaded`).
3. ForgeAgent explores frame-by-frame: ChangeNet ranks actions by predicted change,
   the transition graph prunes loops/no-ops, frontier planning seeks new states —
   and it keeps online-finetuning ChangeNet on *this* game's transitions.

In [ ]:
# Competition environment wheels (torch is preinstalled on Kaggle).
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
# Stage the v19 code + prior from the attached dataset, then sanity-check.
# Auto-discovers the dataset by locating combined_agent.py under /kaggle/input,
# so the exact dataset slug does not matter.
import os, glob, shutil, ast

WORK = '/kaggle/working'
NEEDED   = ['combined_agent.py', 'forge_agent.py']
OPTIONAL = ['pretrained_weights.pt']

hits = glob.glob('/kaggle/input/**/combined_agent.py', recursive=True)
assert hits, "combined_agent.py not found under /kaggle/input — attach the v19 dataset (Add Input)."
SRC = os.path.dirname(hits[0])
print('dataset root:', SRC)

# INTEGRITY GUARD — the stored-answer corpus must NEVER ship. Honest solving only.
banned = glob.glob(os.path.join(SRC, 'solutions')) \
       + glob.glob(os.path.join(SRC, '**', 'solutions'), recursive=True) \
       + glob.glob(os.path.join(SRC, '**', '*_bfs_cache_*.json'), recursive=True)
assert not banned, f"INTEGRITY FAIL: stored-answer artifacts in dataset: {banned}. Remove them before uploading."
print('integrity OK: no stored-answer corpus in the dataset')

for f in NEEDED:
    shutil.copy(os.path.join(SRC, f), os.path.join(WORK, f)); print('staged:', f)
for f in OPTIONAL:
    p = os.path.join(SRC, f)
    if os.path.exists(p):
        shutil.copy(p, os.path.join(WORK, f)); print('staged:', f)
    else:
        print('WARNING: missing', f, '-> black-box prior will be COLD (much weaker).')

# truncation / paste-mangle guard (catches a bad upload in 2 seconds)
for f in NEEDED:
    ast.parse(open(os.path.join(WORK, f)).read()); print('syntax OK:', f)

# the prior must be a non-empty, finite state_dict (the NaN lesson)
wp = os.path.join(WORK, 'pretrained_weights.pt')
if os.path.exists(wp):
    import torch
    sd = torch.load(wp, map_location='cpu', weights_only=True)
    assert len(sd) and all(torch.isfinite(v).all() for v in sd.values() if torch.is_tensor(v)), 'bad weights'
    print(f'weights OK: {len(sd)} tensors, all finite')

# full import smoke (interactive runs only — skip during the scoring rerun)
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    os.environ['V19_STORE_SOLUTIONS'] = '0'   # honest mode even in the smoke test
    import sys; sys.path.insert(0, WORK)
    import importlib, forge_agent, combined_agent
    importlib.reload(forge_agent); importlib.reload(combined_agent)
    fa = forge_agent.ForgeAgent(weights=wp if os.path.exists(wp) else None)
    fa.reset('smoke')
    print('smoke OK: combined_agent + forge_agent import; prior =',
          'loaded' if os.path.exists(wp) else 'COLD')

In [ ]:
import os, ast
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # wait for the evaluation gateway, then stage the official harness
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents

    # entry point: combined_agent.py exposes `class MyAgent` -> stage as my_agent.py
    !cp /kaggle/working/combined_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py
    # forge_agent must be importable by MyAgent: put it on the harness CWD (root,
    # which is sys.path[0]) AND beside my_agent.py (belt and suspenders)
    !cp /kaggle/working/forge_agent.py /kaggle/working/ARC-AGI-3-Agents/forge_agent.py
    !cp /kaggle/working/forge_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/forge_agent.py
    # the prior at BOTH paths MyAgent probes (here=templates dir, and CWD=root)
    !cp /kaggle/working/pretrained_weights.pt /kaggle/working/ARC-AGI-3-Agents/agents/templates/pretrained_weights.pt 2>/dev/null || echo 'WARNING: no weights staged -> black-box prior COLD'
    !cp /kaggle/working/pretrained_weights.pt /kaggle/working/ARC-AGI-3-Agents/pretrained_weights.pt 2>/dev/null || true

    # re-assert EVERYTHING at the point of use (a silent miss would run the agent broken)
    base = '/kaggle/working/ARC-AGI-3-Agents'
    for p in [f'{base}/agents/templates/my_agent.py', f'{base}/forge_agent.py']:
        assert os.path.exists(p), f'STAGING FAILED: {p}'
        ast.parse(open(p).read())
    has_w = os.path.exists(f'{base}/agents/templates/pretrained_weights.pt')
    print('rerun staging verified: my_agent + forge_agent in place; prior =', 'loaded' if has_w else 'COLD')

    # register the agent under the name `myagent`
    with open(f'{base}/agents/__init__.py', 'w') as f:
        f.write('''from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent
load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "myagent": MyAgent}
''')
    with open(f'{base}/.env', 'w') as f:
        f.write('''SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
RECORDINGS_DIR=/kaggle/working/server_recording
''')

    # HONEST run: V19_STORE_SOLUTIONS=0 -> solve live, never hydrate the answer-book.
    # PYTHONUNBUFFERED -> real-time Logs tab; tee -> downloadable artifact.
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg PYTHONUNBUFFERED=1 V19_STORE_SOLUTIONS=0 \
        python main.py --agent myagent 2>&1 | tee /kaggle/working/v19_run.log

The cell above only runs during the competition scoring rerun, not in interactive tests.

**What to look for in the Logs tab (and in `v19_run.log`):**
- `[v19] no white-box source -> black-box fallback (prior=loaded)` — the
  competition-correct path is live and the prior staged. `prior=cold` means
  `pretrained_weights.pt` did not ship — fix the dataset before submitting.
- per game: `BFS: game source not found` (expected on the hidden eval), then a
  stream of `forge` actions.
- level progress: any `levels_completed` increment is a solved level → RHAE points.

**A healthy run = each game ends having spent few actions per level cleared.**
RHAE squares the human/AI action ratio, so a level cleared in 40 actions is worth
far more than the same level cleared in 400. Watch the *actions-to-level*, not just
whether a level was cleared.

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(data=[['1_0', '1', True, 1]],
                              columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)

This is a dummy submission fallback, important to keep.

---

## How to tell whether v19 actually scores well (the learning loop)

1. **Submit.** Save Version → *Save & Run All*; on the leaderboard the score is the
   summed RHAE across the private games. A non-zero score means the **black-box
   ForgeAgent cleared real levels blind** — the only thing that counts in
   competition mode.
2. **Baseline to beat.** The preview winners landed ~6–13%. v18 white-box "solves"
   do **not** transfer (no source on the private set), so judge v19 only against
   other black-box runs.
3. **Read the Logs, not just the number.** Confirm `prior=loaded`; count solved
   levels and the actions each took. If levels clear but cost thousands of actions,
   RHAE will be tiny — the lever is *action-efficiency*, not raw clears.
4. **Improve the prior, re-measure.** Strengthen `pretrained_weights.pt` offline
   (`pretrain.py` / the ExIt flywheel: solve → harvest → `train_wm_v19.py`), check
   held-out **chg-acc** in `WM_LOG.md`, re-upload only the new weights, resubmit.
   Same architecture, better warm start = fewer wasted actions = higher RHAE.
5. **Honest by construction.** `V19_STORE_SOLUTIONS=0` + no `solutions/` in the
   dataset = the score reflects genuine solving. The offline learning reaches the
   hidden games only inside the weights, never as cached answers.